In [ ]:
import re
import os
import time
import subprocess
import tracemalloc
import psutil
from huggingface_hub import InferenceClient
# from google import genai
from openai import OpenAI
from dotenv import load_dotenv
from groq import Groq

def get_file_extension(language):
    extensions = {"python": ".py", "c": ".c", "cpp": ".cpp", "java": ".java", "go": ".go", "rust": ".rs"}
    return extensions.get(language.lower(), ".txt")

def extract_code(llm_output: str) -> str:
    code = llm_output.strip()
    fence_match = re.search(r"```[\w+-]*\n([\s\S]*?)\n```", code)
    if fence_match:
        return fence_match.group(1).strip()
    code = re.sub(r"^```[\w+-]*", "", code)
    code = re.sub(r"```$", "", code)
    return code.strip()

# ... rest of the functions ...

load_dotenv()

# Add this near your other client initializations
# gemini_api_key = os.getenv("GEMINI_API_KEY")
# gemini_client = genai.Client(api_key=gemini_api_key) if gemini_api_key else None

# Initialize API Clients
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
hf_api_key = os.getenv("HF_API_KEY")
hf_client = InferenceClient(token=hf_api_key) if hf_api_key else None

# Initialize GitHub Models using the standard OpenAI library
github_client = OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=os.getenv("GITHUB_API_KEY")
)

# OpenRouter acts exactly like OpenAI, just point it to their base URL
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

def generate_code(prompt, provider, model_id):
    """Universal function to call different LLM APIs dynamically."""
    provider = provider.strip().lower()

    try:
        if provider == "groq":
            response = groq_client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=model_id,
                temperature=0.1,
            )
            return response.choices[0].message.content or ""

        elif provider == "gemini":
            if not gemini_client:
                raise ValueError("Gemini client not initialized. Check your API key.")
            response = gemini_client.models.generate_content(
                model=model_id,
             contents=prompt,
            )
            return response.text or ""
        
        elif provider == "github":
            max_retries = 3
            for attempt in range(max_retries):
                try:
                    response = github_client.chat.completions.create(
                        messages=[{"role": "user", "content": prompt}],
                        model=model_id,
                        temperature=0.1,
                    )
                    return response.choices[0].message.content or ""
                except Exception as e:
                    if "429" in str(e) and attempt < max_retries - 1:
                        print(f"    ⏳ GitHub limit hit. Cooling down for 10s...")
                        time.sleep(10)
                    else:
                        print(f"    ❌ GitHub Error: {str(e)}")
                        return ""
            return ""
        
        elif provider == "openrouter":
            max_retries = 3
            for attempt in range(max_retries):
                try:
                    response = openrouter_client.chat.completions.create(
                        messages=[{"role": "user", "content": prompt}],
                        model=model_id,
                        temperature=0.1,
                        timeout=45.0
                    )
                    # DEFENSIVE CHECK: Prevent NoneType crashes
                    if hasattr(response, 'choices') and response.choices:
                        return response.choices[0].message.content or ""
                    else:
                        print(f"    ⚠️ OpenRouter returned empty object. Retrying...")
                        time.sleep(5)
                except Exception as e:
                    if "429" in str(e) and attempt < max_retries - 1:
                        print(f"    ⚠️ OpenRouter busy. Retrying in 5s (Attempt {attempt + 1})...")
                        time.sleep(5)
                    else:
                        print(f"    ❌ OpenRouter Error: {str(e)}")
                        return ""
            
            # THE FIX: If it tries 3 times and still gets empty objects, safely return empty string
            print(f"    ❌ OpenRouter Failed all retries.")
            return ""

        elif provider == "huggingface":
            if not hf_client:
                raise ValueError("HF_API_KEY not found or client not initialized.")
            
            try:
                # The official SDK handles the URLs, headers, and routing automatically
                response = hf_client.chat_completion(
                    model=model_id,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=2048,
                    temperature=0.1
                )
                return response.choices[0].message.content or ""
            
            except Exception as e:
                error_msg = str(e)
                if "503" in error_msg or "loading" in error_msg.lower():
                    print(f"    ⏳ HF Model {model_id} is currently waking up on their servers. Try again in 20s.")
                else:
                    print(f"    ❌ HF Error: {error_msg}")
                return ""
            
            res = requests.post(url, headers=headers, json=payload, timeout=120)
            if res.status_code == 200:
                return res.json()["choices"][0]["message"]["content"]
            elif res.status_code == 503:
                print("    ⚠️ HF Model is loading, skipping for now...")
                return ""
            else:
                print(f"    ❌ HF Error: {res.status_code} - {res.text}")
                return ""

        else:
            raise ValueError(f"Unsupported provider: {provider}")

    except Exception as e:
        print(f"    ❌ API Error ({provider} - {model_id}): {str(e)}")
        return ""

In [9]:
def get_file_extension(language):
    extensions = {"python": ".py", "c": ".c", "cpp": ".cpp", "java": ".java", "go": ".go", "rust": ".rs"}
    return extensions.get(language.lower(), ".txt")

def extract_code(llm_output: str) -> str:
    # SAFEGUARD: If the API completely fails and returns None, return empty string
    if not llm_output:
        return ""
        
    code = llm_output.strip()
    fence_match = re.search(r"```[\w+-]*\n([\s\S]*?)\n```", code)
    if fence_match:
        return fence_match.group(1).strip()
    code = re.sub(r"^```[\w+-]*", "", code)
    code = re.sub(r"```$", "", code)
    return code.strip()


# New version
import re

def normalize(output):
    """Normalize output by standardizing spaces and floats for cross-language matching."""
    # Convert to lowercase to avoid case-sensitivity issues
    output = output.lower()
    
    # Process line by line
    lines = [line.strip() for line in output.split('\n')]
    
    normalized_lines = []
    for line in lines:
        if not line:
            continue
            
        # Replace multiple spaces with a single space
        line = re.sub(r'\s+', ' ', line)
        
        # Normalize floats: Strip trailing zeros (e.g., 5.00000000 -> 5.0)
        # Keeps precision intact for exact decimals but normalizes padding
        line = re.sub(r'(\d+\.\d*?[1-9])0+\b', r'\1', line)  # 5.1200 -> 5.12
        line = re.sub(r'(\d+)\.0+\b', r'\1.0', line)         # 5.0000 -> 5.0
        
        normalized_lines.append(line)
        
    return '\n'.join(normalized_lines)

def run_test_suite(test_file, legacy_lang, legacy_file,
                   translated_lang, translated_file):

    tests = parse_tests(test_file)

    total = len(tests)
    passed = 0

    for i, input_data in enumerate(tests, 1):

        legacy_output = run_program(
            legacy_lang,
            legacy_file,
            input_data
        )

        translated_output = run_program(
            translated_lang,
            translated_file,
            input_data
        )

        legacy_output = normalize(legacy_output)
        translated_output = normalize(translated_output)

        print(f"\nTest {i}:")

        if translated_output == legacy_output:
            print("  ✅ PASS")
            passed += 1
        else:
            print("  ❌ FAIL")
            print(f"  Expected (Legacy): {legacy_output}")
            print(f"  Translated:        {translated_output}")

    print("\n----------------------------")
    print(f"Passed {passed}/{total} tests")
    print(f"Accuracy: {(passed/total)*100:.2f}%")
    print("----------------------------")


def parse_tests(test_file):
    if not os.path.exists(test_file): return []
    with open(test_file, "r") as f:
        lines = f.readlines()
    if not lines: return []
    t = int(lines[0].strip())
    tests = []
    for i in range(1, t + 1):
        tests.append(lines[i].strip().replace('\\n', '\n') + '\n')
    return tests

def run_program(language, filepath, input_data, timeout=5):
    """A unified runner that returns (output, error_message)."""
    lang = language.lower()
    if not os.path.exists(filepath): return "", "File not found"
    
    try:
        if lang == "python":
            res = subprocess.run(["python", filepath], input=input_data, text=True, capture_output=True, timeout=timeout)
            return res.stdout.strip(), res.stderr if res.returncode != 0 else None
            
        elif lang in ["c", "cpp", "c++"]:
            exe = "./a.out"
            compiler = "gcc" if lang == "c" else "g++"
            comp = subprocess.run([compiler, filepath, "-o", exe], capture_output=True, text=True)
            if comp.returncode != 0: return "", f"Compile error: {comp.stderr}"
            res = subprocess.run([exe], input=input_data, text=True, capture_output=True, timeout=timeout)
            return res.stdout.strip(), res.stderr if res.returncode != 0 else None
            
        # MISSING FORTRAN BLOCK RESTORED HERE
        elif lang in ["fortran", "f77", "f90"]:
            exe = "./a.out"
            comp = subprocess.run(["gfortran", filepath, "-o", exe], capture_output=True, text=True)
            if comp.returncode != 0: return "", f"Compile error: {comp.stderr}"
            res = subprocess.run([exe], input=input_data, text=True, capture_output=True, timeout=timeout)
            return res.stdout.strip(), res.stderr if res.returncode != 0 else None

        elif lang == "java":
            file_dir = os.path.dirname(filepath) or "."
            file_name = os.path.basename(filepath)
            with open(filepath, 'r') as f: java_code = f.read()
            class_match = re.search(r'public\s+class\s+(\w+)', java_code)
            classname = class_match.group(1) if class_match else file_name.replace(".java", "")
            comp = subprocess.run(["javac", file_name], cwd=file_dir, capture_output=True, text=True)
            if comp.returncode != 0: return "", f"Compile error: {comp.stderr}"
            res = subprocess.run(["java", classname], input=input_data, text=True, capture_output=True, timeout=timeout, cwd=file_dir)
            return res.stdout.strip(), res.stderr if res.returncode != 0 else None
            
    except subprocess.TimeoutExpired:
        return "", "Timeout"
    except Exception as e:
        return "", str(e)
    
    return "", f"Language {lang} not supported in runner"

def measure_time_and_memory(language, filepath, input_data, timeout=5):
    """
    Measure execution time and peak memory usage for a program.
    
    Parameters:
    -----------
    language : str
        Programming language ("python", "c", "cpp", "java")
    filepath : str
        Path to source code file
    input_data : str
        Input to provide to the program via stdin
    timeout : int
        Seconds to wait before killing process (default: 5)
    
    Returns:
    --------
    dict with keys:
        - "time": Execution time in milliseconds (float)
        - "memory": Peak memory in MB (float)
        - "output": Program output (str)
        - "error": None if success, error message if failed
    """
    
    if not os.path.exists(filepath):
        return {
            "time": 0,
            "memory": 0,
            "output": "",
            "error": f"File not found: {filepath}"
        }
    
    lang = language.lower()
    
    try:
        if lang == "python":
            # Python: Use tracemalloc to track peak memory
            tracemalloc.start()
            
            start_time = time.time()
            
            result = subprocess.run(
                ["python", filepath],
                input=input_data,
                text=True,
                capture_output=True,
                timeout=timeout
            )
            
            end_time = time.time()
            current, peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()
            
            execution_time_ms = (end_time - start_time) * 1000
            peak_memory_mb = peak / (1024 * 1024)
            
            if result.returncode != 0:
                return {
                    "time": execution_time_ms,
                    "memory": peak_memory_mb,
                    "output": result.stdout,
                    "error": f"Runtime error: {result.stderr}"
                }
            
            return {
                "time": execution_time_ms,
                "memory": peak_memory_mb,
                "output": result.stdout.strip(),
                "error": None
            }
        
        elif lang in ["c", "cpp", "c++"]:
            # C/C++: Compile, then measure live process RSS while running
            exe = "a.out"
            compiler = "gcc" if lang == "c" else "g++"
            
            compile_result = subprocess.run(
                [compiler, filepath, "-o", exe],
                capture_output=True,
                text=True,
                timeout=timeout
            )
            
            if compile_result.returncode != 0:
                return {
                    "time": 0,
                    "memory": 0,
                    "output": "",
                    "error": f"Compilation error: {compile_result.stderr}"
                }
            
            start_time = time.time()
            proc = subprocess.Popen(
                ["./" + exe],
                stdin=subprocess.PIPE,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True
            )
            
            peak_memory_mb = 0.0
            poll_interval = 0.02
            input_once = input_data
            stdout, stderr = "", ""
            
            while True:
                elapsed = time.time() - start_time
                remaining = timeout - elapsed
                if remaining <= 0:
                    proc.kill()
                    return {
                        "time": timeout * 1000,
                        "memory": peak_memory_mb,
                        "output": "",
                        "error": f"Timeout after {timeout}s"
                    }
                
                try:
                    stdout, stderr = proc.communicate(input=input_once, timeout=min(poll_interval, remaining))
                    break
                except subprocess.TimeoutExpired:
                    input_once = None
                    try:
                        rss_mb = psutil.Process(proc.pid).memory_info().rss / (1024 * 1024)
                        peak_memory_mb = max(peak_memory_mb, rss_mb)
                    except (psutil.NoSuchProcess, psutil.AccessDenied):
                        pass
            
            end_time = time.time()
            execution_time_ms = (end_time - start_time) * 1000
            
            if proc.returncode != 0:
                return {
                    "time": execution_time_ms,
                    "memory": peak_memory_mb,
                    "output": stdout,
                    "error": f"Runtime error: {stderr}"
                }
            
            return {
                "time": execution_time_ms,
                "memory": peak_memory_mb,
                "output": stdout.strip(),
                "error": None
            }
        
        # ... [existing C/C++ block] ...
        
        elif lang in ["fortran", "f77", "f90", "f95"]:
            # Fortran: Compile, then measure live process RSS while running
            exe = "a.out"
            
            compile_result = subprocess.run(
                ["gfortran", filepath, "-o", exe],
                capture_output=True,
                text=True,
                timeout=timeout
            )
            
            if compile_result.returncode != 0:
                return {
                    "time": 0, "memory": 0, "output": "",
                    "error": f"Compilation error: {compile_result.stderr}"
                }
            
            start_time = time.time()
            proc = subprocess.Popen(
                ["./" + exe],
                stdin=subprocess.PIPE,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True
            )
            
            peak_memory_mb = 0.0
            poll_interval = 0.02
            input_once = input_data
            stdout, stderr = "", ""
            
            while True:
                elapsed = time.time() - start_time
                remaining = timeout - elapsed
                if remaining <= 0:
                    proc.kill()
                    return {
                        "time": timeout * 1000, "memory": peak_memory_mb, 
                        "output": "", "error": f"Timeout after {timeout}s"
                    }
                
                try:
                    stdout, stderr = proc.communicate(input=input_once, timeout=min(poll_interval, remaining))
                    break
                except subprocess.TimeoutExpired:
                    input_once = None
                    try:
                        # psutil tracks the memory of the running Fortran binary
                        rss_mb = psutil.Process(proc.pid).memory_info().rss / (1024 * 1024)
                        peak_memory_mb = max(peak_memory_mb, rss_mb)
                    except (psutil.NoSuchProcess, psutil.AccessDenied):
                        pass
            
            end_time = time.time()
            execution_time_ms = (end_time - start_time) * 1000
            
            if proc.returncode != 0:
                return {"time": execution_time_ms, "memory": peak_memory_mb, "output": stdout, "error": f"Runtime error: {stderr}"}
            
            return {"time": execution_time_ms, "memory": peak_memory_mb, "output": stdout.strip(), "error": None}

        # ... [existing Java block] ...
        
        elif lang == "java":
            # Java: Compile, then measure
            file_dir = os.path.dirname(filepath) if os.path.dirname(filepath) else "."
            file_name = os.path.basename(filepath)
            
            with open(filepath, 'r') as f:
                java_code = f.read()
            
            class_match = re.search(r'public\s+class\s+(\w+)', java_code)
            classname = class_match.group(1) if class_match else file_name.replace(".java", "")
            
            compile_result = subprocess.run(
                ["javac", file_name],
                cwd=file_dir,
                capture_output=True,
                text=True,
                timeout=timeout
            )
            
            if compile_result.returncode != 0:
                return {
                    "time": 0,
                    "memory": 0,
                    "output": "",
                    "error": f"Compilation error: {compile_result.stderr}"
                }
            
            start_time = time.time()
            proc = subprocess.Popen(
                ["java", classname],
                stdin=subprocess.PIPE,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                cwd=file_dir
            )
            
            try:
                stdout, stderr = proc.communicate(input=input_data, timeout=timeout)
                end_time = time.time()
            except subprocess.TimeoutExpired:
                proc.kill()
                return {
                    "time": timeout * 1000,
                    "memory": 0,
                    "output": "",
                    "error": f"Timeout after {timeout}s"
                }
            
            execution_time_ms = (end_time - start_time) * 1000
            peak_memory_mb = 0
            
            if proc.returncode != 0:
                return {
                    "time": execution_time_ms,
                    "memory": peak_memory_mb,
                    "output": stdout,
                    "error": f"Runtime error: {stderr}"
                }
            
            return {
                "time": execution_time_ms,
                "memory": peak_memory_mb,
                "output": stdout.strip(),
                "error": None
            }
        
        else:
            return {
                "time": 0,
                "memory": 0,
                "output": "",
                "error": f"Language '{language}' not supported. Use: python, c, cpp, java"
            }
    
    except subprocess.TimeoutExpired:
        return {
            "time": timeout * 1000,
            "memory": 0,
            "output": "",
            "error": f"Timeout after {timeout}s"
        }
    except Exception as e:
        return {
            "time": 0,
            "memory": 0,
            "output": "",
            "error": f"Exception: {str(e)}"
        }

    pass

In [10]:
import json
import os
import re

def run_benchmark(provider, model_id, inp_lng, target_lng, input_path, folder_path, test_file):
    print(f"\n" + "="*80)
    print(f"🚀 STARTING BENCHMARK: {model_id} ({provider.upper()})")
    print("="*80)

    safe_model_id = model_id.replace("/", "_").replace("-", "_")
    base_name = os.path.splitext(os.path.basename(input_path))[0]
    ext = get_file_extension(target_lng)
    
    results = {
        "model": model_id, "provider": provider, "translation_status": "Failed",
        "correction_attempts": 0, "passed_tests": False, 
        "legacy_time": 0.0, "legacy_memory": 0.0,   # <--- ADD THESE TWO
        "initial_time": 0.0, "initial_memory": 0.0, "opt_v1_time": 0.0, "best_improvement": 0.0
    }

    with open(input_path, "r") as f:
        input_code = f.read()
        
    if not input_code.strip():
        print(f"❌ FATAL ERROR: The input file ({input_path}) is empty!")
        return results

    # ---------------------------------------------------------
    # --- PHASE 1: TRANSLATION ---
    # ---------------------------------------------------------
    print("\n--- Phase 1: Translation ---")
    
    prompt = f'''You are an exceptionally intelligent coding assistant acting as a literal syntax compiler for code translation. You consistently deliver accurate translations while maintaining the original code's exact functionality, output format, and numerical precision.

Please translate this {inp_lng} code to {target_lng}. Follow these CRITICAL guidelines:

1) **COMPLETE PROGRAM**: Translate the ENTIRE program including ALL subroutines/functions/methods, initialization, calculations, and display/output routines. Do NOT omit any functions or return partial code. The output must be a complete, executable program.

2) **LITERAL MATHEMATICS (NO HALLUCINATIONS)**: Translate mathematical operations, formulas, and variable assignments EXACTLY as they appear in the source code. 
   - DO NOT substitute formulas from your own knowledge. 
   - Preserve all numeric constants and precision multipliers exactly.
   - Example: `0.5 * dt * dx` stays as `0.5 * dt * dx` (do not simplify differently).

3) **EXACT I/O & PROMPTS**: Every input must produce IDENTICAL output byte-for-byte in strict test environments.
   - Read inputs dynamically in the same order as the original. NEVER hardcode test values.
   - Preserve every single print statement, prompt string, and delimiter exactly.
   - Do NOT add extra newlines unless they exist in the source. Handle line-by-line input correctly.

4) **DISPLAY/PRINT LOOPS WITH STRIDE**: When translating loops that print values with a stride pattern (e.g., printing every nth element):
   - Preserve the stride/step calculation precisely.
   - DO NOT print every single value if the original prints selectively—this guarantees test failures.
   - Translate format specifications to exact target equivalents (e.g., Fortran `format(f10.4,' | ',f15.2)` becomes Python `print(f"{{val1:10.4f}} | {{val2:15.2f}}")`).

5) **OUTPUT FORMAT**: Preserve EXACT formatting:
   - Decimal precision (2 decimals, 4 decimals, etc.).
   - Field widths, padding, string formatting, and alignment.

6) **ARRAYS & INDEXING (CRITICAL BOUNDS)**: 
   - Identify the source language's indexing system (e.g., 1-based vs 0-based).
   - Convert array access appropriately for the target language.
   - strictly adjust loop boundaries to prevent off-by-one errors. If a source loop runs exactly 19 times, the target loop MUST run exactly 19 times.

7) **STRUCTURE**: 
   - Maintain overall organization (subroutines/functions/methods, main logic/entry point).
   - Use appropriate language constructs (classes if needed for state management).
   - Keep method/function names and parameter names clear and meaningful.

8) **NUMERICAL OPERATIONS**: 
   - Preserve numeric types (integer, float, double precision).
   - Preserve integer division semantics strictly according to the source language rules.
   - Preserve floating-point precision and rounding behavior.

9) **LANGUAGE-SPECIFIC CONVERSIONS**: 
   - Translate language constructs appropriately (e.g., COMMON blocks to classes, modules to namespaces).
   - Convert language-specific keywords and syntax to exact target language equivalents.

10) **LIBRARIES & IMPORTS**: Include all necessary imports/headers/libraries needed for the algorithms to run.

11) **OUTPUT**: Return ONLY the complete translated code in a single code block. NO explanations, NO extra text, NO incomplete snippets.

12) **DO NOT**: 
    - Optimize or refactor algorithms.
    - Change output stride/sampling patterns.
    - Omit any functions, subroutines, or methods.
    - Change the program flow or control structure.
13) **STRICT I/O SEPARATION**: 
    - In Python, NEVER put prompt strings inside the `input()` function (e.g., NO `input("Enter value:")`).
    - You MUST separate prompts and inputs. Use `print("Enter value:")` followed by `val = input()`. This perfectly mimics Fortran's `print *` newline behavior.
    
14) **NO HELPFUL TEXT**: Do not add helpful text like `(y/n)` or `...` to the prompts. Translate the strings exactly as they appear in the original source, character for character.
15) MULTI-VARIABLE INPUTS (CRITICAL): Fortran's read(*,*) a, b reads tokens across multiple newlines. Python's input().split() only reads a single line. You MUST perfectly mimic Fortran by printing the original prompt string ONCE, followed by separate input() calls for each variable.
BAD: a, b = input("Enter a and b: ").split()
GOOD: > print("Enter a and b: ")
a = type(input().strip())
b = type(input().strip())

"16) **DIVISION BY ZERO (IEEE 754)**: Fortran natively handles division by zero by evaluating to `Infinity` and propagating `NaN`s, whereas Python crashes with `ZeroDivisionError`. If a formula may divide by zero, you MUST handle it by returning `float('inf')` (or `-inf` based on the numerator) to perfectly match Fortran's behavior. NEVER add an epsilon (e.g., `+ 1e-6`) or return 0, as this causes numerical explosions and OverflowErrors in physics loops.\n",

Code to Translate: {input_code}
'''
    
    raw_output = generate_code(prompt, provider, model_id)
    translated_code = extract_code(raw_output)

    if not translated_code:
        print("❌ Translation failed (Empty response or error).")
        return results

    current_filepath = os.path.join(folder_path, f"{base_name}_{safe_model_id}_translated{ext}")
    with open(current_filepath, "w") as f: f.write(translated_code)
    results["translation_status"] = "Success"
    print(f"✅ Translation complete. Saved to {current_filepath}")

    # ---------------------------------------------------------
    # --- PHASE 2: ERROR CORRECTION ---
    # ---------------------------------------------------------
    print("\n--- Phase 2: Error Correction ---")
    tests = parse_tests(test_file)
    max_attempts = 5
    all_passed = False

    for attempt in range(max_attempts):
        feedback = []
        for i, input_data in enumerate(tests, 1):
            legacy_out, _ = run_program(inp_lng, input_path, input_data)
            translated_out, err = run_program(target_lng, current_filepath, input_data)
            
            # CRITICAL FIX: Must use normalize() here to ignore Fortran vs Python spacing differences
            norm_legacy = normalize(legacy_out)
            norm_translated = normalize(translated_out)
            
            if err or norm_legacy != norm_translated:
                def truncate_text(text, limit=1000):
                    if not text: return ""
                    if len(text) <= limit: return text
                    return text[:200] + "\n... [TRUNCATED] ...\n" + text[-800:]

                feedback.append({
                    "test": i, 
                    "input": input_data.replace('\n', '\\n'), 
                    "expected": truncate_text(norm_legacy), 
                    "actual": truncate_text(norm_translated) if not err else f"ERROR: {truncate_text(err)}"
                })

        if not feedback:
            print(f"✅ All tests passed on attempt {attempt + 1}!")
            all_passed = True
            break
        
        print(f"❌ Attempt {attempt + 1}: {len(feedback)} tests failed. Attempting fix...")
        results["correction_attempts"] += 1
        
        # RESTORED DEBUG PRINTER
        # ENHANCED DEBUG PRINTER (SHOWS EXACT DIFF)
        if attempt == 0:
            import difflib
            print(f"\n [DEBUG] EXACT ERROR FOR TEST 1:")
            exp_lines = feedback[0]['expected'].split('\n')
            act_lines = feedback[0]['actual'].split('\n')
            
            print("\n--- EXPECTED (Fortran) vs ACTUAL (Python) DIFF ---")
            print("(- indicates missing in Python, + indicates extra in Python)")
            diff = '\n'.join(difflib.ndiff(exp_lines, act_lines))
            print(diff)
            print("🛑" * 20 + "\n")

        nav_prompt = f'''You are an expert code translation debugging assistant. Analyze failed test cases and identify the ROOT CAUSE of differences between the {inp_lng} and {target_lng} code.

ANALYZE THE FOLLOWING FATAL ERRORS:
1. **Dropped I/O**: Did the code skip reading a variable entirely?
2. **NameError**: Did a subroutine try to accept variables that weren't defined yet?
3. **SyntaxError (Global/Parameter)**: Did the code declare a variable as `global` while also passing it as a function parameter?
4. **ValueError (unpacking)**: Did Python use `.split()` on a single line when inputs are on multiple lines?
5. **Off-By-One Bounds**: Did a loop start at 1 instead of 0, skipping the initial state?
6. **Concatenated Prompts**: Did the diff show multiple prompts merged onto one line (e.g., `+ prompt 1 -> prompt 2 ->`)? This means the LLM incorrectly used `input("prompt")` instead of `print()`.
7. **Missing Data Rows**: Did Python print the first and last values but skip the middle ones? Check the display loop's stride calculation (e.g., `max(1, n//20)`).

Provide STRICT JSON with actionable fix hints:
{{
  "error_summary": "...",
  "affected_lines": "...",
  "root_cause": "...",
  "fix_hints": ["hint 1", "hint 2"]
}}

Failed Test Case(s):
{feedback}

Legacy Code ({inp_lng}):
{input_code}

Translated Code ({target_lng}):
{translated_code}
'''
        print(f"Navigator is analyzing... (Attempt {attempt + 1})")
        navigator_raw = generate_code(nav_prompt, provider, model_id)
        
        if not navigator_raw:
            print("Navigator API returned empty response. Retrying...")
            continue
            
      
        json_match = re.search(r'\{[\s\S]*\}', navigator_raw)
        navigator_analysis = json_match.group(0) if json_match else navigator_raw.strip()
        
        print(f"Driver is applying fix...")
        driver_prompt = f'''The translated {target_lng} code failed the test cases against the original {inp_lng} code.
**Original Legacy Code ({inp_lng}):**
{input_code}
**Current Translated Code to Fix ({target_lng}):**
{translated_code}
Navigator Analysis:
{navigator_analysis}

Failed Tests:
{feedback}

CRITICAL FIX REQUIREMENTS:
1. **Apply Navigator Fixes**: Implement the exact logic and boundary fixes identified by the Navigator.
2. **DO NOT HARDCODE VALUES**: The code must continue to read from standard input and process dynamic test cases. Hardcoding to pass a specific test is strictly prohibited.
3. **PRESERVE ENTIRE STRUCTURE**: You must return the FULL, runnable program, including all imports, initializations, standard input reading, and printing. Do not omit or truncate any functions.
4. **EXACT FUNCTIONALITY**: Do not change the underlying algorithm or overall structure.
5. **BYTE-FOR-BYTE MATCH**: Fix the output formatting, precision, spacing, and loop bounds so the output matches the expected test output identically.
6. FIXING VALUEERRORS (I/O): If you see ValueError: not enough values to unpack, it means the test case provided inputs on multiple newlines. Fix this by explicitly reading across multiple lines (e.g., calling input().strip() sequentially). DO NOT split the original print() statement into multiple new prompt strings. You must print the exact original string once, then capture the variables.
7. INTERLEAVED I/O RULE: Legacy Fortran frequently interleaves reading from the terminal (read(5,*)) and reading from external files (open(1...), read(1,*)). When translating to Python, or when suggesting fixes for Python input handling, you MUST NOT use sys.stdin.read(), sys.stdin.readlines(), or sys.stdin iterators. You must use sequential input().strip() calls for terminal inputs to ensure the standard input buffer is perfectly preserved while the external file is being read.

Return ONLY the fully corrected, executable {target_lng} code in a single code block. NO explanations, NO markdown outside the code block.'''
        
        candidate_code = extract_code(generate_code(driver_prompt, provider, model_id))
        
        if not candidate_code:
            print("    ⚠️ Driver API failed or returned empty code. Skipping overwrite.")
            continue
            
        translated_code = candidate_code
        with open(current_filepath, "w") as f:
            f.write(translated_code)

    results["passed_tests"] = all_passed
    if not all_passed:
        print("Max correction attempts reached. Skipping optimization.")
        return results

    # ---------------------------------------------------------
    # --- PHASE 3: PERFORMANCE & OPTIMIZATION ---
    # ---------------------------------------------------------
    print("\n--- Phase 3: Performance & Optimization ---")
    
    # --- NEW: Measure Legacy Code First ---
    print(f"\nMeasuring Legacy Code ({inp_lng}) Performance...")
    legacy_res_list = [measure_time_and_memory(inp_lng, input_path, inp) for inp in tests]
    successful_legacy = [r for r in legacy_res_list if not r['error']]
    
    if len(successful_legacy) > 0:
        results["legacy_time"] = sum(r['time'] for r in successful_legacy) / len(successful_legacy)
        results["legacy_memory"] = sum(r['memory'] for r in successful_legacy) / len(successful_legacy)
        print(f"  Legacy Time:   {results['legacy_time']:.3f} ms")
        print(f"  Legacy Memory: {results['legacy_memory']:.3f} MB")
    else:
        print("Legacy code failed to execute during profiling.")
    # --------------------------------------

    prev_time = None
    
    
        # ... [rest of your Phase 3 loop remains exactly the same] ...
    for iteration in range(4):
        if iteration == 0:
            print(f"\nBaseline Performance (Initial Translated Code)")
        else:
            print(f"\nOptimization Iteration {iteration}/3")
        print("-" * 60)

        res_list = [measure_time_and_memory(target_lng, current_filepath, inp) for inp in tests]

        successful_tests = [r for r in res_list if not r['error']]
    
    # DEFENSIVE CHECK: Prevent ZeroDivisionError if no tests pass or if the test list is empty
        if len(successful_tests) > 0:
            avg_t = sum(r['time'] for r in successful_tests) / len(successful_tests)
            avg_m = sum(r['memory'] for r in successful_tests) / len(successful_tests)
        else:
            print("    ⚠️ No successful tests to average. Setting performance metrics to 0.")
            avg_t = 0.0
            avg_m = 0.0
        # avg_t = sum(r['time'] for r in res_list if not r['error']) / len(tests)
        # avg_m = sum(r['memory'] for r in res_list if not r['error']) / len(tests)

        # SAFEGUARD RESTORED: Reject broken optimizations
        if iteration > 0 and (avg_t == 0.0 or any(r['error'] for r in res_list)):
            print("LLM broke the code during optimization. Discarding this version.")
            break

        if iteration == 0:
            results["initial_time"] = avg_t
            results["initial_memory"] = avg_m
            print(f"  Baseline Time:   {avg_t:.3f} ms")
            print(f"  Baseline Memory: {avg_m:.3f} MB")
        else:
            if iteration == 1:
                results["opt_v1_time"] = avg_t

            improvement_pct = ((prev_time - avg_t) / prev_time * 100) if prev_time > 0 else 0
            print(f"  Current Time:    {avg_t:.3f} ms")
            print(f"  Current Memory:  {avg_m:.3f} MB")
            print(f"  vs Previous:     {improvement_pct:.2f}% improvement")
            
            total_improvement = ((results["initial_time"] - avg_t) / results["initial_time"] * 100) if results["initial_time"] > 0 else 0
            results["best_improvement"] = max(results["best_improvement"], total_improvement)

            if avg_t >= prev_time * 0.95:
                print("No significant improvement. Stopping optimization.")
                break
        
        prev_time = avg_t
        if iteration == 3: break

        print(f"\nRequesting Optimization V{iteration + 1} from LLM...")
        opt_prompt = f"Optimize this {target_lng} code for speed. Return ONLY code inside a markdown block.\nCode:\n{translated_code}"
        translated_code = extract_code(generate_code(opt_prompt, provider, model_id))
        
        opt_filepath = os.path.join(folder_path, f"{base_name}_{safe_model_id}_opt_v{iteration+1}{ext}")
        with open(opt_filepath, "w") as f:
            f.write(translated_code)
        current_filepath = opt_filepath

    print(f"\n🏁 Benchmark completed for {model_id}\n" + "="*80)
    return results

In [ ]:
import uuid

folder_path = "Sample_Codes/Physics_Sample_Codes/Discrete_Maps/"
input_path = os.path.join(folder_path, "discrete_maps.f")
test_file = os.path.join(folder_path, "tests.txt")

input_code = open(input_path, "r").read()

thread_id = str(uuid.uuid4())

result = app.invoke(
    {
        "input_code": input_code,
        "inp_lang": "fortran",
        "target_lang": "python",
    },
    config={
        "configurable": {
            "thread_id": thread_id,
            "provider": "groq",
            "model_id": "llama-3.3-70b-versatile",
            "tests": parse_tests(test_file),
        }
    }
)

print(result["translated_code"])
print(f"Passed: {result['passed']}, Attempts used: {result['attempt_count']}")


🚀 STARTING BENCHMARK: gpt-4o-mini (GITHUB)

--- Phase 1: Translation ---
    ❌ GitHub Error: Error code: 401 - {'error': {'code': 'unauthorized', 'message': 'Bad credentials', 'details': 'Bad credentials'}}
❌ Translation failed (Empty response or error).

🚀 STARTING BENCHMARK: gpt-4o (GITHUB)

--- Phase 1: Translation ---
    ❌ GitHub Error: Error code: 401 - {'error': {'code': 'unauthorized', 'message': 'Bad credentials', 'details': 'Bad credentials'}}
❌ Translation failed (Empty response or error).

🚀 STARTING BENCHMARK: llama-3.3-70b-versatile (GROQ)

--- Phase 1: Translation ---
✅ Translation complete. Saved to Sample_Codes/Physics_Sample_Codes/Discrete_Maps/discrete_maps_llama_3.3_70b_versatile_translated.c

--- Phase 2: Error Correction ---
✅ All tests passed on attempt 1!

--- Phase 3: Performance & Optimization ---

Measuring Legacy Code (fortran) Performance...
  Legacy Time:   329.940 ms
  Legacy Memory: 0.031 MB

Baseline Performance (Initial Translated Code)
--------------

KeyboardInterrupt: 

In [ ]:
print("\n" + "╔" + "═"*165 + "╗")
print(f"║ {'LLM COMPREHENSIVE PERFORMANCE & OPTIMIZATION DETAIL':^163} ║")
print("╠" + "═"*35 + "╦" + "═"*8 + "╦" + "═"*7 + "╦" + "═"*11 + "╦" + "═"*10 + "╦" + "═"*11 + "╦" + "═"*10 + "╦" + "═"*10 + "╦" + "═"*13 + "╦" + "═"*11 + "╣")
print(f"║ {'MODEL NAME':<33} ║ {'TRANS.':^6} ║ {'FIXES':^5} ║ {'LEGACY TM':^9} ║ {'LEGACY M':^8} ║ {'INIT TIME':^9} ║ {'INIT MEM':^8} ║ {'OPT V1':^8} ║ {'BEST IMPR.':^11} ║ {'STATUS':^9} ║")
print(f"║ {'':<33} ║ {'P1':^6} ║ {'P2':^5} ║ {'(ms)':^9} ║ {'(MB)':^8} ║ {'(ms)':^9} ║ {'(MB)':^8} ║ {'(ms)':^8} ║ {'(%)':^11} ║ {'':^9} ║")
print("╠" + "═"*35 + "╬" + "═"*8 + "╬" + "═"*7 + "╬" + "═"*11 + "╬" + "═"*10 + "╬" + "═"*11 + "╬" + "═"*10 + "╬" + "═"*10 + "╬" + "═"*13 + "╬" + "═"*11 + "╣")

for r in all_results:
    p1 = "✅" if r['translation_status'] == "Success" else "❌"
    p2 = f"{r['correction_attempts']}f" if r['passed_tests'] else "FAIL"
    
    # Grab Legacy metrics
    lt = f"{r.get('legacy_time', 0.0):.2f}" if r.get('legacy_time', 0) > 0 else "—"
    lm = f"{r.get('legacy_memory', 0.0):.3f}" if r.get('legacy_time', 0) > 0 else "—"
    
    # Grab Init/Opt metrics
    t0 = f"{r['initial_time']:.2f}" if r['passed_tests'] else "—"
    m0 = f"{r['initial_memory']:.3f}" if r['passed_tests'] else "—"
    t1 = f"{r['opt_v1_time']:.2f}" if r.get('opt_v1_time') and r['opt_v1_time'] > 0 else "—"
    
    imp = r.get('best_improvement', 0.0)
    imp_str = f"{imp:+.2f}%" if imp != 0 else "0.00%"
    
    status = "SUCCESS" if r['passed_tests'] else "FAILED"

    print(f"║ {r['model'][:33]:<33} ║ {p1:^6} ║ {p2:^5} ║ {lt:^9} ║ {lm:^8} ║ {t0:^9} ║ {m0:^8} ║ {t1:^8} ║ {imp_str:^11} ║ {status:^9} ║")

print("╚" + "═"*35 + "╩" + "═"*8 + "╩" + "═"*7 + "╩" + "═"*11 + "╩" + "═"*10 + "╩" + "═"*11 + "╩" + "═"*10 + "╩" + "═"*10 + "╩" + "═"*13 + "╩" + "═"*11 + "╝")


╔═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                                                         LLM COMPREHENSIVE PERFORMANCE & OPTIMIZATION DETAIL                                                         ║
╠═══════════════════════════════════╦════════╦═══════╦═══════════╦══════════╦═══════════╦══════════╦══════════╦═════════════╦═══════════╣
║ MODEL NAME                        ║ TRANS. ║ FIXES ║ LEGACY TM ║ LEGACY M ║ INIT TIME ║ INIT MEM ║  OPT V1  ║ BEST IMPR.  ║  STATUS   ║
║                                   ║   P1   ║  P2   ║   (ms)    ║   (MB)   ║   (ms)    ║   (MB)   ║   (ms)   ║     (%)     ║           ║
╠═══════════════════════════════════╬════════╬═══════╬═══════════╬══════════╬═══════════╬══════════╬══════════╬═════════════╬═══════════╣
║ gpt-4o-mini                       ║   ✅    ║  2f   ║  343.42   ║  0.099   ║  114.62   ║  0.100   ║  105.06  ║

In [ ]:
# from huggingface_hub import list_models

# print("Fetching top downloaded text-generation models from Hugging Face...\n")

# try:
#     # Changed sort parameter to 'downloads'
#     active_models = list_models(
#         filter="text-generation",
#         sort="downloads",
#         limit=20
#     )

#     print("✅ TOP DOWNLOADED TEXT MODELS (Most likely to be active):\n" + "-"*50)
#     for model in active_models:
#         print(f"\"{model.id}\"")
        
# except Exception as e:
#     print(f"Error fetching models: {e}")

In [ ]:
# import requests

# def get_active_free_models():
#     response = requests.get("https://openrouter.ai/api/v1/models")
#     if response.status_code == 200:
#         all_models = response.json()['data']
#         # Filter for models that are free (price is 0)
#         free_models = [m['id'] for m in all_models if float(m['pricing']['prompt']) == 0]
#         print("✅ CURRENTLY ACTIVE FREE MODELS ON OPENROUTER:")
#         for model in free_models[:15]: # Show top 15
#             print(f"- {model}")
#     else:
#         print("Failed to fetch catalog.")

# get_active_free_models()

In [ ]:
# import requests

# github_token = os.getenv("GITHUB_API_KEY") # Paste your ghp_... token here
# url = "https://models.inference.ai.azure.com/models"

# headers = {
#     "Authorization": f"Bearer {github_token}",
#     "Content-Type": "application/json"
# }

# try:
#     response = requests.get(url, headers=headers)
#     if response.status_code == 200:
#         models = response.json()
#         print("✅ ACTIVE GITHUB MODELS:")
#         # Azure returns a list of dictionaries directly
#         for model in models:
#             # Safely grab the model name/id
#             model_id = model.get("name") or model.get("id")
#             print(f"- {model_id}")
#     else:
#         print(f"❌ Error {response.status_code}: {response.text}")
# except Exception as e:
#     print(f"❌ Connection Error: {e}")